#  IMPORTS & ENVIRONMENT SETUP

In [ ]:
# Cell 1 + TEST: Restart kernel → chạy
!pip install -U ultralytics -q

import sys
from pathlib import Path
import torch.nn as nn

# GSConv class
from ultralytics.nn.modules.conv import Conv
from ultralytics.nn.modules import DWConv

class GSConv(nn.Module):
    def __init__(self, c1, c2, k=1, s=1, g=1, act=True):
        super().__init__()
        c_ = c2 // 2
        self.cv1 = Conv(c1, c_, k, s, g=g, act=False)
        self.cv2 = DWConv(c_, c_, k=5, s=1, act=act)

    def forward(self, x):
        x1 = self.cv1(x)
        x2 = self.cv2(x1)
        B, C, H, W = x2.shape
        x_cat = torch.cat([x1, x2], dim=1)
        return x_cat.view(B, 2, C//2, H, W).permute(0,2,1,3,4).contiguous().view(B, C, H, W)

# REGISTER VÀO TASK GLOBALS
import ultralytics.nn.tasks
ultralytics.nn.tasks.GSConv = GSConv

# TEST LOAD
from ultralytics import YOLO
model = YOLO("yolov8_dgs_fixed.yaml")
print("✅ LOADED THÀNH CÔNG! Layers:", len(model.model.model))
print("Sẵn sàng train!")

In [ ]:
# Cell 2 FIXED: YOLOv8-DGS YAML CHUẨN ULTRALYTICS
yaml_content = """
# YOLOv8-DGS cho RDD26k - FORMAT CHUẨN YOLOv8
nc: 5  # classes 0,1,2,3,4 từ RDD2020
names: ['0', '1', '2', '3', '4']

# BẮT BUỘC: scales cho YOLOv8
scales:
  n: [0.33, 0.25, 1024]  # nano

backbone:
  # DWConv backbone layers 0,1,3,5 (paper Fig.2)
  - [-1,  1, DWConv, [32,  3, 2]]  # 0-P1/2
  - [-1,  1, DWConv, [64,  3, 2]]  # 1-P2/4
  - [-1,  3,  C2f,  [64, True]]
  - [-1,  1, DWConv, [128, 3, 2]]  # 3-P3/8
  - [-1,  6,  C2f,  [128, True]]
  - [-1,  1, DWConv, [256, 3, 2]]  # 5-P4/16
  - [-1,  6,  C2f,  [256, True]]
  - [-1,  1,  SPPF, [256, 5]]      # 9

head:
  - [-1, 1, nn.Upsample, [None, 2, 'nearest']]
  - [[-1, 6], 1, Concat, [1]]      # cat backbone P4 (layer 6)
  - [-1, 3, C2f, [128]]            # 12

  - [-1, 1, nn.Upsample, [None, 2, 'nearest']]
  - [[-1, 4], 1, Concat, [1]]      # cat backbone P3 (layer 4)
  - [-1, 3, C2f, [64]]             # 15 (P3/8-small)

  - [-1, 1, GSConv, [64, 3, 2]]    # 16-GSConv paper neck
  - [[-1, 12], 1, Concat, [1]]     # cat head layer 12
  - [-1, 3, C2f, [128]]            # 18 (P4/16-medium)

  - [-1, 1, GSConv, [128, 3, 2]]   # 19-GSConv paper neck
  - [[-1, 9], 1, Concat, [1]]      # cat backbone SPPF layer 9
  - [-1, 3, C2f, [256]]            # 21 (P5/32-large)

  # Detect(P3=15, P4=18, P5=21) - indices ĐÚNG!
  - [[15, 18, 21], 1, Detect, [nc]]
"""

with open("yolov8_dgs_fixed.yaml", "w") as f:
    f.write(yaml_content)
print("✅yolov8_dgs_fixed.yaml - FORMAT CHUẨN YOLOv8!")

## PATHS 

In [ ]:
# 1_setup.py — chạy đầu tiên
import os
from pathlib import Path
import torch
import cv2
import numpy as np

# Đường dẫn của bạn
BDD_ROOT = Path(r"D:\Code\RDD26k")
TRAIN_IMG_DIR = BDD_ROOT / "train" / "images"
TRAIN_LBL_DIR = BDD_ROOT / "train" / "labels"
VALID_IMG_DIR = BDD_ROOT / "valid" / "images"
VALID_LBL_DIR = BDD_ROOT / "valid" / "labels"
TEST_IMG_DIR = BDD_ROOT / "test" / "images"
TEST_LBL_DIR = BDD_ROOT / "test" / "labels"
DATA_YAML_PATH = BDD_ROOT / "data.yaml"

# Kiểm tra dataset
print("=== KIỂM TRA DATASET ===")
print(f"Train images: {len(list(TRAIN_IMG_DIR.glob('*.jpg')))}")
print(f"Train labels: {len(list(TRAIN_LBL_DIR.glob('*.txt')))}")
print(f"Valid images: {len(list(VALID_IMG_DIR.glob('*.jpg')))}")
print(f"Valid labels: {len(list(VALID_LBL_DIR.glob('*.txt')))}")
print(f"Test images:  {len(list(TEST_IMG_DIR.glob('*.jpg')))}")

# Đọc data.yaml
with open(DATA_YAML_PATH) as f:
    yaml_content = f.read()
print("\nDATA.YAML:")
print(yaml_content)

# Kiểm tra 1 label mẫu (class index 0-4?)
sample_lbl = next(TRAIN_LBL_DIR.glob("*.txt"))
with open(sample_lbl) as f:
    lines = f.readlines()
print(f"\nSample label {sample_lbl.name}:")
print(lines[:3])  # 3 dòng đầu

# Kiểm tra GPU
print(f"\nCUDA available: {torch.cuda.is_available()}")
print("Setup OK!")

In [ ]:
# Cell 2: YOLOv8-DGS YAML (nc=5 RDD2020)
yaml_content = """
# YOLOv8-DGS Nano cho RDD26k (DWConv backbone + GSConv neck)
nc: 5  # classes 0,1,2,3,4
names: ['0', '1', '2', '3', '4']

backbone:
  # DWConv thay Conv ở layers 0,1,3,5 (paper)
  - [-1, 1, DWConv, [32, 3, 2]]     # 0-P1/2
  - [-1, 1, DWConv, [64, 3, 2]]     # 1-P2/4  
  - [-1, 3, C2f, [64, True]]
  - [-1, 1, DWConv, [128, 3, 2]]    # 3-P3/8
  - [-1, 6, C2f, [128, True]]
  - [-1, 1, DWConv, [256, 3, 2]]    # 5-P4/16
  - [-1, 6, C2f, [256, True]]
  - [-1, 1, SPPF, [256, 5]]         # 9

head:
  - [-1, 1, nn.Upsample, [None, 2, 'nearest']]
  - [[-1, 6], 1, Concat, [1]] 
  - [-1, 3, C2f, [128]]             # 12
  - [-1, 1, nn.Upsample, [None, 2, 'nearest']]
  - [[-1, 4], 1, Concat, [1]] 
  - [-1, 3, C2f, [64]]              # 15-P3/8

  - [-1, 1, GSConv, [64, 3, 2]]     # 16-GSConv paper neck
  - [[-1, 12], 1, Concat, [1]] 
  - [-1, 3, C2f, [128]]             # 18-P4/16

  - [-1, 1, GSConv, [128, 3, 2]]    # 19-GSConv paper neck
  - [[-1, 9], 1, Concat, [1]] 
  - [-1, 3, C2f, [256]]             # 21-P5/32

  - [[15, 18, 21], 1, Detect, [nc]]
"""

with open("yolov8_dgs.yaml", "w") as f:
    f.write(yaml_content)
print("✅ yolov8_dgs.yaml created!")

In [ ]:
# 2_preprocess_full.py — chạy FULL train set
import cv2
import numpy as np
from pathlib import Path
import time

BDD_ROOT = Path(r"D:\Code\RDD26k")
TRAIN_IMG_DIR = BDD_ROOT / "train" / "images"

def filter_equalization(img, T_thresh=1.8):
    filtered = cv2.bilateralFilter(img, d=4, sigmaColor=80, sigmaSpace=75)
    gray = cv2.cvtColor(filtered, cv2.COLOR_BGR2GRAY).astype(float)
    mu = np.mean(gray)
    sigma = np.std(gray)
    T = sigma / (mu + 1e-6)
    
    if T < T_thresh:
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        lab = cv2.cvtColor(filtered, cv2.COLOR_BGR2LAB)
        lab[:,:,0] = clahe.apply(lab[:,:,0])
        return cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)
    return filtered

print("🚀 FILTER EQUALIZATION - 18,771 images (~5-10p)")
start_time = time.time()

count = 0
for img_path in TRAIN_IMG_DIR.glob("*.jpg"):
    img = cv2.imread(str(img_path))
    if img is None: continue
    
    enhanced = filter_equalization(img)
    cv2.imwrite(str(img_path), enhanced, [cv2.IMWRITE_JPEG_QUALITY, 95])
    count += 1
    if count % 1000 == 0:
        print(f"Processed: {count}/{18771}")

print(f"✅ Done! {time.time()-start_time:.1f}s")

In [ ]:
# Cell 2: YOLOv8-DGS YAML (nc=5 RDD2020)
yaml_content = """
# YOLOv8-DGS Nano cho RDD26k (DWConv backbone + GSConv neck)
nc: 5  # classes 0,1,2,3,4
names: ['0', '1', '2', '3', '4']

backbone:
  # DWConv thay Conv ở layers 0,1,3,5 (paper)
  - [-1, 1, DWConv, [32, 3, 2]]     # 0-P1/2
  - [-1, 1, DWConv, [64, 3, 2]]     # 1-P2/4  
  - [-1, 3, C2f, [64, True]]
  - [-1, 1, DWConv, [128, 3, 2]]    # 3-P3/8
  - [-1, 6, C2f, [128, True]]
  - [-1, 1, DWConv, [256, 3, 2]]    # 5-P4/16
  - [-1, 6, C2f, [256, True]]
  - [-1, 1, SPPF, [256, 5]]         # 9

head:
  - [-1, 1, nn.Upsample, [None, 2, 'nearest']]
  - [[-1, 6], 1, Concat, [1]] 
  - [-1, 3, C2f, [128]]             # 12
  - [-1, 1, nn.Upsample, [None, 2, 'nearest']]
  - [[-1, 4], 1, Concat, [1]] 
  - [-1, 3, C2f, [64]]              # 15-P3/8

  - [-1, 1, GSConv, [64, 3, 2]]     # 16-GSConv paper neck
  - [[-1, 12], 1, Concat, [1]] 
  - [-1, 3, C2f, [128]]             # 18-P4/16

  - [-1, 1, GSConv, [128, 3, 2]]    # 19-GSConv paper neck
  - [[-1, 9], 1, Concat, [1]] 
  - [-1, 3, C2f, [256]]             # 21-P5/32

  - [[15, 18, 21], 1, Detect, [nc]]
"""

with open("yolov8_dgs.yaml", "w") as f:
    f.write(yaml_content)
print("✅ yolov8_dgs.yaml created!")

In [ ]:
# Cell 3: TRAINING YOLOv8-DGS - GIỐNG HỆT PAPER
from ultralytics import YOLO
import torch

device = 0 if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

# Load YOLOv8-DGS (DWConv+GSConv theo paper Fig.2)
print("🔄 Loading YOLOv8-DGS (DWConv backbone + GSConv neck)...")
model = YOLO("yolov8_dgs.yaml")

# Hyperparameters GIỐNG PAPER Section 4.3
print("🚀 Training giống paper (RDD2020 → mAP50 ~91.6%)...")
results = model.train(
    # Dataset
    data=str(DATA_YAML_PATH),
    
    # Paper settings
    epochs=200,                # Table 3
    imgsz=640,
    batch=32,                  # 16 nếu OOM
    
    # Optimizer (AdamW paper dùng)
    device=device,
    project="runs_dgs_paper",
    name="yolov8_dgs_rdd26k",
    exist_ok=True,
    pretrained=True,           # yolov8n.pt init
    
    # Augmentation (paper tắt Mosaic + low aug)
    mosaic=0.0,                # Filter Equalization thay thế
    mixup=0.0,
    copy_paste=0.0,
    fliplr=0.5,
    flipud=0.0,
    
    # Learning rate schedule (paper Fig.6)
    lr0=0.01,                  # initial LR
    lrf=0.02,                  # final LR = lr0 * lrf
    momentum=0.937,
    weight_decay=0.0005,       # L2 reg paper
    
    # Warmup (paper)
    warmup_epochs=5,
    warmup_momentum=0.8,
    warmup_bias_lr=0.1,
    
    # Loss weights (paper ablation)
    box=7.5,
    cls=0.5,
    dfl=1.5,
    
    # Save & monitor
    save=True,
    save_period=20,
    plots=True,
    val=True,
    patience=50,               # early stopping
    
    # Speed (paper 85 FPS target)
    workers=8,
    amp=True  # Auto mixed precision
)

print("🎉 Training hoàn thành giống paper!")
print("Best model: runs_dgs_paper/yolov8_dgs_rdd26k/weights/best.pt")